# Follow water conditions through time

Select one micro-watershed and compare annual groundwater change, fortnightly rainfall/ET/runoff, and the area of mapped surface waterbodies.

Run each cell with **Shift+Enter**. The location controls default to the active KYL tehsil when this notebook is downloaded from CoRE Stack.

In [ ]:
import json, re, sys
from urllib.parse import urlencode
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
import geolibre

GEOSERVER_BASE = "https://geoserver.core-stack.org:8443/geoserver/"
SCOPE = json.loads("{\"state\":\"Jharkhand\",\"district\":\"Dumka\",\"tehsil\":\"Masalia\",\"bounds\":[86.89,23.94,87.24,24.28]}")
LAYER_SPECS = json.loads("[{\"id\":\"mws_layers\",\"label\":\"Micro-watersheds and Hydrological Variables\",\"domain\":\"Hydrology\",\"service\":\"WFS\",\"workspace\":\"mws_layers\",\"layerNameTemplate\":\"deltaG_well_depth_{district}_{tehsil}\",\"period\":\"2017-2018 to 2024-2025\",\"description\":\"Annual groundwater-storage change and MWS identifiers.\"},{\"id\":\"mws_layers_fortnight\",\"label\":\"Fortnightly Hydrological Variables\",\"domain\":\"Hydrology\",\"service\":\"WFS\",\"workspace\":\"mws_layers\",\"layerNameTemplate\":\"deltaG_fortnight_{district}_{tehsil}\",\"period\":\"July 2017 to June 2025\",\"description\":\"Fortnightly precipitation, evapotranspiration, runoff, and related water-balance values.\"},{\"id\":\"remote_sensed_waterbodies\",\"label\":\"Remote-Sensed Waterbodies\",\"domain\":\"Hydrology\",\"service\":\"WFS\",\"workspace\":\"swb\",\"layerNameTemplate\":\"surface_waterbodies_{district}_{tehsil}\",\"period\":\"2017-2018 to 2024-2025\",\"description\":\"Waterbody extent, seasonal area, use, ownership, storage, and beneficiaries.\"}]")
m = geolibre.connect()
MAP_LAYERS = {}

def geoserver_name(value):
    value = re.sub(r"[()]", "", str(value or "").strip().lower())
    return re.sub(r"_+", "_", re.sub(r"\s+", "_", value)).strip("_")

state_input = widgets.Text(value=SCOPE["state"], description="State:", layout=widgets.Layout(width="98%"))
district_input = widgets.Text(value=SCOPE["district"], description="District:", layout=widgets.Layout(width="98%"))
tehsil_input = widgets.Text(value=SCOPE["tehsil"], description="Tehsil:", layout=widgets.Layout(width="98%"))
display(widgets.VBox([
    widgets.HTML("<b>Study location</b><br><small>Change a name here, then rerun the data cells. No Python editing is needed.</small>"),
    state_input, district_input, tehsil_input,
]))

def selected_scope():
    return {
        "state": state_input.value.strip(),
        "district": geoserver_name(district_input.value),
        "tehsil": geoserver_name(tehsil_input.value),
    }

def get_spec(layer_id):
    return next(layer for layer in LAYER_SPECS if layer["id"] == layer_id)

def layer_url(layer_id, cql_filter=None):
    scope = selected_scope()
    spec = get_spec(layer_id)
    layer_name = spec["layerNameTemplate"].format(**scope)
    qualified = f'{spec["workspace"]}:{layer_name}'
    if spec["service"] == "WFS":
        params = {"service": "WFS", "version": "1.0.0", "request": "GetFeature",
                  "typeName": qualified, "outputFormat": "application/json", "srsName": "EPSG:4326"}
        if cql_filter:
            params["CQL_FILTER"] = cql_filter
        return f'{GEOSERVER_BASE}{spec["workspace"]}/ows?{urlencode(params)}'
    params = {"service": "WCS", "version": "2.0.1", "request": "GetCoverage",
              "CoverageId": qualified, "format": "geotiff", "compression": "LZW"}
    return f'{GEOSERVER_BASE}{spec["workspace"]}/wcs?{urlencode(params)}'

async def fetch_json(url, label="GeoServer layer"):
    try:
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            response = await pyfetch(url)
            if not response.ok:
                raise RuntimeError(f"HTTP {response.status}")
            return await response.json()
        import urllib.request
        with urllib.request.urlopen(url, timeout=90) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception as error:
        scope = selected_scope()
        raise RuntimeError(
            f'{label} is not available for {scope["district"]}/{scope["tehsil"]}, or GeoServer could not be reached: {error}'
        ) from error

async def load_geojson(layer_id, cql_filter=None):
    spec = get_spec(layer_id)
    if spec["service"] != "WFS":
        raise ValueError(f'{spec["label"]} is a raster. Use its WCS URL instead of loading it as GeoJSON.')
    data = await fetch_json(layer_url(layer_id, cql_filter), spec["label"])
    if data.get("type") != "FeatureCollection":
        raise RuntimeError(f'{spec["label"]} did not return GeoJSON features.')
    return data

def to_frame(data):
    rows = [dict(feature.get("properties") or {}) for feature in data.get("features", [])]
    return pd.DataFrame(rows)

def uid_column(frame):
    for name in ("uid", "UID", "MWS_UID", "MWS UID"):
        if name in frame.columns:
            return name
    raise KeyError("This layer has no recognised MWS identifier column.")

def with_uid(frame):
    result = frame.copy()
    result["uid"] = result[uid_column(result)].astype(str)
    return result

def numeric(frame, columns):
    return frame.loc[:, columns].apply(pd.to_numeric, errors="coerce")

def json_component(value, component):
    try:
        value = json.loads(value) if isinstance(value, str) else value
        return float(value.get(component)) if isinstance(value, dict) and value.get(component) is not None else np.nan
    except (TypeError, ValueError, json.JSONDecodeError):
        return np.nan

def component_values(frame, columns, component):
    return frame.loc[:, columns].apply(
        lambda series: series.map(lambda value: json_component(value, component))
    )

def features_for_uids(data, uids):
    wanted = {str(uid) for uid in uids}
    names = ("uid", "UID", "MWS_UID", "MWS UID")
    features = []
    for feature in data.get("features", []):
        properties = feature.get("properties") or {}
        value = next((properties.get(name) for name in names if properties.get(name) is not None), None)
        if str(value) in wanted:
            features.append(feature)
    return {"type": "FeatureCollection", "features": features}

def geojson_bounds(data):
    points = []
    def visit(value):
        if isinstance(value, list) and len(value) >= 2 and all(isinstance(v, (int, float)) for v in value[:2]):
            points.append(value[:2])
        elif isinstance(value, list):
            for item in value:
                visit(item)
    for feature in data.get("features", []):
        visit((feature.get("geometry") or {}).get("coordinates", []))
    if not points:
        return None
    xs, ys = zip(*points)
    return [min(xs), min(ys), max(xs), max(ys)]

def show_on_map(key, data, name, **style):
    if not data.get("features"):
        print(f"No features to map for {name}.")
        return None
    previous_layer_id = MAP_LAYERS.get(key)
    if previous_layer_id:
        try:
            m.remove_layer(previous_layer_id)
        except Exception:
            pass
    MAP_LAYERS[key] = m.add_geojson(data, name=name, **style)
    bounds = geojson_bounds(data)
    if bounds:
        m.fit_bounds(bounds)
    return MAP_LAYERS[key]

def year_columns(frame, prefix="", pattern=r"^\d{4}_\d{4}$"):
    return sorted(column for column in frame.columns if column.startswith(prefix) and re.search(pattern, column))

print(f'Ready for {SCOPE["tehsil"]}, {SCOPE["district"]}.')

In [ ]:
def annual_groundwater(frame, mws_id):
    row = with_uid(frame).set_index("uid").loc[str(mws_id)]
    columns = sorted(column for column in frame.columns if re.match(r"^\d{4}_\d{4}$", column))
    return pd.DataFrame({"Year": [column.replace("_", "-") for column in columns],
                         "Groundwater change": [json_component(row[column], "DeltaG") for column in columns]})

def fortnightly_balance(frame, mws_id):
    row = with_uid(frame).set_index("uid").loc[str(mws_id)]
    records = []
    for column in sorted(name for name in frame.columns if re.match(r"^\d{4}-\d{2}-\d{2}$", name)):
        value = row[column]
        try:
            value = json.loads(value) if isinstance(value, str) else value
            records.append({"Date": pd.to_datetime(column), "Precipitation": float(value.get("Precipitation") or 0),
                            "ET": float(value.get("ET") or 0), "Runoff": float(value.get("RunOff") or 0)})
        except (TypeError, ValueError, json.JSONDecodeError):
            continue
    return pd.DataFrame(records)

def waterbody_history(frame):
    columns = sorted(column for column in frame.columns if re.match(r"^area_\d{2}-\d{2}$", column))
    values = numeric(frame, columns).sum(axis=0) if len(frame) else pd.Series(index=columns, dtype=float)
    return pd.DataFrame({"Year": [column.replace("area_", "20", 1) for column in columns],
                         "Mapped waterbody area": values.values})

def plot_hydrology(annual, fortnight, waterbodies, mws_id):
    fig, axes = plt.subplots(3, 1, figsize=(11, 10))
    axes[0].plot(annual["Year"], annual["Groundwater change"], marker="o", color="#1d4ed8")
    axes[0].axhline(0, color="#991b1b", linewidth=1)
    axes[0].set(title=f"Annual groundwater change · {mws_id}", ylabel="Published change")
    for field, color in [("Precipitation", "#2563eb"), ("ET", "#ea580c"), ("Runoff", "#16a34a")]:
        axes[1].plot(fortnight["Date"], fortnight[field], label=field, color=color, linewidth=1)
    axes[1].set(title="Fortnightly water-balance components", ylabel="mm")
    axes[1].legend(ncol=3)
    axes[2].plot(waterbodies["Year"], waterbodies["Mapped waterbody area"], marker="o", color="#0891b2")
    axes[2].set(title="Surface-water area inside this MWS", ylabel="Published area", xlabel="Hydrological year")
    plt.tight_layout()
    plt.show()

## 1. Load the MWS histories

Only annual and fortnightly MWS layers are loaded initially. Waterbodies are requested after an MWS is selected.

In [ ]:
annual_geojson = await load_geojson("mws_layers")
fortnight_geojson = await load_geojson("mws_layers_fortnight")
annual_frame = to_frame(annual_geojson)
fortnight_frame = to_frame(fortnight_geojson)
mws_picker = widgets.Dropdown(options=sorted(with_uid(annual_frame)["uid"]), description="MWS:")
display(mws_picker)

## 2. Plot the selected MWS

Choose an MWS above, then rerun this cell. The waterbody request is filtered to that MWS to keep the download small.

In [ ]:
mws_id = mws_picker.value
water_geojson = await load_geojson("remote_sensed_waterbodies", f"MWS_UID='{mws_id}'")
annual_series = annual_groundwater(annual_frame, mws_id)
fortnight_series = fortnightly_balance(fortnight_frame, mws_id)
waterbody_series = waterbody_history(to_frame(water_geojson))
plot_hydrology(annual_series, fortnight_series, waterbody_series, mws_id)

## 3. Locate the selected MWS

The highlighted polygon is temporary and does not alter CoRE Stack data.

In [ ]:
selected = features_for_uids(annual_geojson, [mws_id])
show_on_map(
    "selected-hydrology-mws", selected, f"Notebook · {mws_id}",
    fillColor="#22d3ee", strokeColor="#164e63", fillOpacity=0.55,
)

## Interpretation

Groundwater-storage change, rainfall, ET, runoff, and mapped surface-water area describe different parts of the water system. Their co-movement can motivate questions, but it does not establish that one series caused another.

## Optional: inspect the underlying fortnightly table

In [ ]:
display(fortnight_series.tail(26).round(2))